# 作业 3：构建 NDArray 库

在本作业中，你将为大多数深度学习系统底层的处理构建一个简单的支撑库：n 维数组（即 NDArray）。到目前为止，你基本上一直在使用 numpy 来完成此目的，但本作业将引导你开发自己的（尽管功能更有限的）numpy 变体，它将支持 CPU 和 GPU 后端。更重要的是，与 numpy（甚至像 PyTorch 这样的变体）不同，你不会简单地调用现有的高度优化的矩阵乘法或其他操作代码，而是编写自己的版本，这些版本与支撑这些标准库的高度优化代码具有合理的竞争力（在某种程度上，即'仅慢 2-3 倍'……这比容易慢 100 倍的朴素代码好得多）。这个类最终将集成到 `needle` 中，但对于本作业，你_只需_关注 ndarray 模块，因为这将是测试的唯一主题。

**注意**：为避免在 Colab 中耗尽有限的 GPU 资源，请先使用 CPU 运行时来编码和测试非 GPU 函数。在测试 CUDA 或 GPU 加速代码时再切换到 GPU 运行时。这种方法可确保高效的 GPU 使用，并防止在关键任务期间资源耗尽。

In [1]:
# 设置作业环境
import sys, os, importlib
os.chdir(os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd())
sys.path.insert(0, './python')

def reload_nd():
    """修改 Python/C++ 源码后，运行此函数重新加载模块"""
    # 清除编译的后端模块缓存（.so 文件会在下次 import 时重新加载）
    for mod_name in list(sys.modules.keys()):
        if 'ndarray_backend' in mod_name or 'backend_ndarray' in mod_name:
            del sys.modules[mod_name]
    # 重载 Python 模块
    import needle.backend_ndarray.ndarray
    import needle.backend_selection
    importlib.reload(needle.backend_ndarray.ndarray)
    importlib.reload(needle.backend_selection)
    # 清除测试模块缓存
    for mod_name in list(sys.modules.keys()):
        if 'test_' in mod_name:
            del sys.modules[mod_name]
    print('模块已重载，可以调试')

# 带超时的测试运行器
import subprocess, sys
def run_test(query, timeout=10):
    """运行测试，带超时和详细输出"""
    cmd = [sys.executable, "-m", "pytest", "-v", "-s", "-k", query, "--tb=long", "--no-header"]
    try:
        result = subprocess.run(cmd, timeout=timeout, capture_output=True, text=True, cwd=os.getcwd())
        print(result.stdout)
        if result.stderr: print("STDERR:", result.stderr)
    except subprocess.TimeoutExpired as e:
        print(f"!!! TIMEOUT ({timeout}s) !!!")
        if e.stdout: print(e.stdout)
        if e.stderr: print("STDERR:", e.stderr)


In [2]:
# 修改源码后运行此单元格（C++ 代码需先执行 !make）
reload_nd()

Using needle backend
Using needle backend
模块已重载，可以调试


In [3]:
# REQUIRED FOR MUGRADE
MY_API_KEY = "<FILL YOUR API KEY HERE>"
HW3_NAME = "hw3"

In [4]:
# 编译 C++/CUDA 后端
!export PATH="/home/x1879/miniconda3/envs/dlsys/nvvm/bin:$PATH" && make

-- Found pybind11: /home/x1879/miniconda3/envs/dlsys/lib/python3.13/site-packages/pybind11/include (found version "3.0.4")
-- Found cuda, building cuda backend
-- Detected CUDA architecture: 8.9
-- Configuring done (0.2s)
-- Generating done (0.0s)
-- Build files have been written to: /home/x1879/dlsys/hws/hw3/build
make[1]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[2]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
[ 50%] Built target ndarray_backend_cpu
make[3]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
[ 75%] Building CUDA object CMakeFiles/ndarray_backend_cuda.dir/src/ndarray_backend_cuda.cu.o
[100%] Linking CUDA shared module /home/x1879/dlsys/hws/hw3/python/needle/backend_ndarray/ndarray_backend_cuda.cpytho

make 命令读取当前目录中的 Makefile。Makefile 包含定义如何构建目标（如可执行文件或库）的规则。对于 Makefile 中指定的每个目标，make 会检查目标文件及其依赖项（如 .c、.cpp 或 .h 文件）的时间戳。如果任何依赖项最近被修改过，就必须重新构建目标。

In [5]:
%set_env PYTHONPATH ./python
%set_env NEEDLE_BACKEND nd

env: PYTHONPATH=./python
env: NEEDLE_BACKEND=nd


In [6]:
import sys
sys.path.append('./python')

## 熟悉 NDArray 类

开始本作业时，你应首先熟悉我们提供的 `NDArray.py` 类，它作为作业的起点。代码相当简短（约 500 行，但其中很多是为你要实现的函数提供的注释）。

NDArray 类的核心是处理通用 n 维数组操作的 Python 包装器。回想一下，几乎任何这样的数组在内部都存储为浮点值的向量，即

```c++
float data[size];
```

然后对数组不同维度的实际访问都由额外的字段（如数组形状、步长等）处理，这些字段指示这个'扁平'数组如何映射到 n 维结构。为了达到任何合理的速度，'原始'操作（如加法、二元运算，以及更结构化的操作如矩阵乘法等）都需要在某种程度上用 C++ 等原生语言编写（包括例如进行 CUDA 调用）。但大量的操作如转置、广播、矩阵子集等，都可以仅通过调整数组的高层结构（如步长）来处理。

NDArray 类背后的理念是，我们希望_所有_处理这种数组结构的逻辑都用 Python 编写。只有在扁平向量上执行原始底层操作的'真正'低级代码（以及管理这些扁平向量的代码，因为它们可能需要例如在 GPU 上分配）才用 C++ 编写。这种分离的精确性质在你完成作业时可能会变得更清楚，但一般来说，所有能在 Python 中完成的事情都在 Python 中完成；通常以一些效率低下为代价……我们频繁调用 `.compact()`（复制内存）以使底层 C++ 实现更简单。

更详细地说，NDArray 类中有五个你需要熟悉的字段（注意这些字段的实际类成员前面都有下划线，例如 `_handle`、`_strides` 等，其中一些作为公共属性暴露……对于你所有的代码，使用内部带下划线的版本是可以的）。

1. `device` - `BackendDevice` 类型的对象，它是一个简单的包装器，包含指向底层设备后端（如 CPU 或 CUDA）的链接。
2. `handle` - 存储数组底层内存的类对象。它被分配为 `device.Array()` 类型，不过这种分配都发生在提供的代码中（具体是 `NDArray.make` 函数），你不需要自己调用它。
3. `shape` - 指定数组每个维度大小的元组。
4. `strides` - 指定数组每个维度步长的元组。
5. `offset` - 一个整数，指示数组在底层 `device.Array` 内存中的实际起始位置（存储这个很方便，这样我们可以更容易地管理指向现有内存的指针，而不需要跟踪分配）。

通过操作这些字段，即使是纯 Python 代码也能执行很多需要的数组操作，如排列维度（即转置）、广播等。对于原始数组操作调用，`device` 类有许多方法（用 C++ 实现）包含必要的实现。

有几点需要注意：

* 在内部，类可以使用_任何_高效的方式来操作数据数组作为'设备'后端。甚至可以使用 numpy 数组，但不是实际使用 `numpy.ndarray` 来表示 n 维数组，而是在 numpy 中表示一个'扁平'的 1D 数组，然后调用相关的 numpy 方法来实现这个原始内存上的所有需要的操作。这正是我们在 `ndarray_backend_numpy.py` 文件中所做的，它本质上提供了一个'存根参考'，对所有东西都使用 numpy。你可以使用这个类来帮助你更好地调试自己的'真正' CPU 和 GPU 后端实现。
* 对于你的许多 Python 实现特别重要的是 `NDArray.make` 调用：
```python
def make(shape, strides=None, device=None, handle=None, offset=0):
```
它创建一个具有给定 shape、strides、device、handle 和 offset 的新 NDArray。如果未指定 `handle`（即未引用预先存在的内存），则调用将分配所需的内存，但如果指定了 handle，则不会分配新内存，而是新 NDArray 指向与旧 NDArray 相同的内存。对于高效实现，你的函数尽可能_不_分配新内存是很重要的，因此你会想在很多情况下使用这个调用来实现这一点。
* NDArray 有一个 `.numpy()` 调用，将数组转换为 numpy。这与'numpy_device'后端不同：这创建一个等效于给定 NDArray 的实际 `numpy.ndarray`，即相同的维度、形状等，但步长不一定相同（Pybind11 将以这种方式返回的矩阵重新分配内存，这可能会改变步长）。

## 第 1 部分：Python 数组操作

作为类的起点，在 `ndarray.py` 文件中实现以下函数：

- `reshape()`
- `permute()`
- `broadcast_to()`
- `__getitem__()`

这些函数的输入/输出都在函数存根的文档字符串中描述。需要强调的是，这些函数_都不应该_重新分配内存，而应该返回与 `self` 共享相同内存的 NDArray，只是巧妙地操作 shape/strides 等来获得必要的变换。

开始时你可以参考课堂幻灯片中提到的关于转置、广播和切片运算符的提示。

有一点需要注意，`__getitem__()` 调用与 numpy 不同，它永远不会改变数组的维度数。因此，例如，对于 2D NDArray `A`，`A[2,2]` 将返回一个仍然 2D 的数组，只有一行一列。例如 `A[:4,2]` 将返回一个 4 行 1 列的 2D NDArray。

你可以依赖 `ndarray_backend_numpy.py` 模块来完成本节的所有代码。你也可以查看等效 numpy 操作的结果（测试用例应该说明这些是什么）。

实现这些函数后，你应该通过/提交以下测试。注意我们在下面的测试中测试了所有这四个函数，你可以随着实现每个额外的函数逐步尝试通过更多测试。

In [7]:
import pytest
reload_nd()
pytest.main(['-v', '-k', '(permute or reshape or broadcast or getitem) and cpu and not compact'])

Using needle backend
模块已重载，可以调试
============================= test session starts ==============================
platform linux -- Python 3.13.13, pytest-9.0.3, pluggy-1.6.0 -- /home/x1879/miniconda3/envs/dlsys/bin/python
cachedir: .pytest_cache
rootdir: /home/x1879/dlsys/hws/hw3
plugins: timeout-2.4.0
collecting ... collected 136 items / 126 deselected / 10 selected

tests/hw3/test_ndarray.py::test_permute[cpu-params0] PASSED              [ 10%]
tests/hw3/test_ndarray.py::test_permute[cpu-params1] PASSED              [ 20%]
tests/hw3/test_ndarray.py::test_permute[cpu-params2] PASSED              [ 30%]
tests/hw3/test_ndarray.py::test_reshape[cpu-params0] PASSED              [ 40%]
tests/hw3/test_ndarray.py::test_reshape[cpu-params1] PASSED              [ 50%]
tests/hw3/test_ndarray.py::test_getitem[cpu-params0] PASSED              [ 60%]
tests/hw3/test_ndarray.py::test_getitem[cpu-params1] PASSED              [ 70%]
tests/hw3/test_ndarray.py::test_getitem[cpu-params2] PASSED          

<ExitCode.OK: 0>

In [8]:
# 提交到 mugrade（需要 API key）
# 取消注释以下行并替换 MY_API_KEY
# !python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_python_ops"

1 2 3 4 5 6
1 2 3
4 5 6  (2,3)(3,1)
1 4
2 5
4 6  (3,2)(1,3)



## 第 2 部分：CPU 后端 - Compact 和 setitem

在 `ndarray_backend_cpu.cc` 中实现以下函数：
* `Compact()`
* `EwiseSetitem()`
* `ScalarSetitem()`

要了解为什么这些都在同一类别中，让我们考虑 `Compact()` 函数的实现。回想一下，如果矩阵以'行优先'形式在内存中顺序排列（但实际上是对更高维数组的行优先的推广），即最后的维度在前，然后是倒数第二个维度，依此类推，一直到第一个维度，则该矩阵被认为是紧凑的。在我们的实现中，我们还要求分配的后端数组的总大小等于数组的大小（即底层数组在数组数据之前或之后也不能有任何数据，例如这意味着 `.offset` 字段等于零）。

现在让我们以 3D 数组为例，考虑 compact 调用如何工作。这里 `shape` 和 `strides` 是被压缩矩阵的形状和步长（即在我们压缩它之前）。

```c++
cnt = 0;
for (size_t i = 0; i < shape[0]; i++)
    for (size_t j = 0; j < shape[1]; j++)
        for (size_t k = 0; k < shape[2]; k++)
            out[cnt++] = in[strides[0]*i + strides[1]*j + strides[2]*k];
```
换句话说，我们正在将矩阵从基于步长的表示转换为纯顺序表示。

现在，实现 `Compact()` 的挑战在于你希望该方法适用于任意数量的输入维度。为不同的固定维度大小特化很容易，但对于通用实现，你需要考虑如何执行相同的操作，实际上你想要的是'可变数量的 for 循环'。作为提示，一种方法是维护一个索引向量（大小等于维度数），然后在循环中手动递增它们（当任何一个达到最大大小时包括'进位'操作）。

但是，如果你真的卡住了，你可以利用我们可能不会要求你处理超过 6 维矩阵的事实（尽管我们_会_使用 6 维，用于我们在课堂上讨论的 im2col 操作）。


#### 与 setitem 的联系
setitem 功能虽然看起来非常不同，但实际上与 `Compact()` 密切相关。`__setitem()__` 是设置对象某些元素时调用的 Python 函数，即
```python
A[::2,4:5,9] = 0 # 或 = some_other_array
```
你将如何实现这一点？在上面的 `__getitem()__` 调用中，你已经实现了在不复制的情况下取矩阵子集的方法（只是修改步长）。但你将如何实际_设置_这个数组的元素？在本作业中几乎所有其他设置中，我们在设置输出数组中的元素之前调用 `.compact()`，但在这种情况下它不起作用：调用 `.compact()` 会将子集数组复制到某些新内存，但 `__setitem__()` 调用的全部意义在于我们想要修改现有内存。

如果你思考一段时间，你会意识到答案看起来很像 `.compact()` 但反过来。如果我们想要将一个（本身已经是紧凑的）右手边矩阵赋值给 `__getitem()__` 的结果，那么我们需要这里的 `shape` 和 `stride` 是_输出_矩阵的那些字段。然后我们可以如下实现 setitem 调用

```c++
cnt = 0;
for (size_t i = 0; i < shape[0]; i++)
    for (size_t j = 0; j < shape[1]; j++)
        for (size_t k = 0; k < shape[2]; k++)
            out[strides[0]*i + strides[1]*j + strides[2]*k] = in[cnt++]; // 或 "= val;"
```
由于这种相似性，如果你以模块化方式实现索引策略，你将能够在 `Compact()` 调用和 `EwiseSetitem()` 和 `ScalarSetitem()` 调用之间重用它。

**注意**：在执行测试之前不要忘记运行 make 来重新构建你修改过的 C++ 代码。

1 2 3
4 5 6
7 8 9 (3,3)(3,1)0

8   8%3*1+(8-8%3)/3%3*3

4 6
7 9(2,2)(3,2)3

3  3%2*2+

In [9]:
!export PATH="/home/x1879/miniconda3/envs/dlsys/nvvm/bin:$PATH" && make
reload_nd()
run_test('(compact or setitem) and cpu', timeout=10)

-- Found pybind11: /home/x1879/miniconda3/envs/dlsys/lib/python3.13/site-packages/pybind11/include (found version "3.0.4")
-- Found cuda, building cuda backend
-- Detected CUDA architecture: 8.9
-- Configuring done (0.1s)
-- Generating done (0.0s)
-- Build files have been written to: /home/x1879/dlsys/hws/hw3/build
make[1]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[2]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
[ 50%] Built target ndarray_backend_cpu
make[3]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
[100%] Built target ndarray_backend_cuda
make[2]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
make[1]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
Using needle backend
模块已重载，可以调试
============================= test session starts ============================

In [10]:
# 提交到 mugrade（需要 API key）
# 取消注释以下行并替换 MY_API_KEY
# !python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cpu_compact_setitem"

## 第 3 部分：CPU 后端 - 逐元素和标量操作

在 `ndarray_backend_cpu.cc` 中实现以下函数：

* `EwiseMul()`, `ScalarMul()`
* `EwiseDiv()`, `ScalarDiv()`
* `ScalarPower()`
* `EwiseMaximum()`, `ScalarMaximum()`
* `EwiseEq()`, `ScalarEq()`
* `EwiseGe()`, `ScalarGe()`
* `EwiseLog()`
* `EwiseExp()`
* `EwiseTanh()`

你可以查看包含的 `EwiseAdd()` 和 `ScalarAdd()` 函数（加上从 `NDArray` 的调用）来理解这些函数所需的格式。

注意与这里提到的其余函数不同，我们不为每个函数提供函数存根。这是因为虽然你可以通过单独实现每个函数来朴素地实现这些，但这最终会有大量重复代码。欢迎你使用例如 C++ 模板或宏来解决这个问题（但这些只能在内部暴露，不能暴露给外部接口）。

In [11]:
import pytest
!export PATH="/home/x1879/miniconda3/envs/dlsys/nvvm/bin:$PATH" && make
reload_nd()
run_test('(ewise_fn or ewise_max or log or exp or tanh or (scalar and not setitem)) and cpu')

-- Found pybind11: /home/x1879/miniconda3/envs/dlsys/lib/python3.13/site-packages/pybind11/include (found version "3.0.4")
-- Found cuda, building cuda backend
-- Detected CUDA architecture: 8.9
-- Configuring done (0.1s)
-- Generating done (0.0s)
-- Build files have been written to: /home/x1879/dlsys/hws/hw3/build
make[1]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[2]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
[ 50%] Built target ndarray_backend_cpu
make[3]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
[100%] Built target ndarray_backend_cuda
make[2]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
make[1]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
Using needle backend
模块已重载，可以调试
============================= test session starts ============================

In [12]:
# 提交到 mugrade（需要 API key）
# 取消注释以下行并替换 MY_API_KEY
# !python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cpu_ops"

## 第 4 部分：CPU 后端 - 归约操作


在 `ndarray_backend_cpu.cc` 中实现以下函数：

* `ReduceMax()`
* `ReduceSum()`

一般来说，NDArray 中的归约函数 `.max()` 和 `.sum()` 沿 `axis` 参数指定的指定轴取最大值或求和（当 `axis=None` 时沿整个数组）；注意我们不支持 axis 是一组轴，不过如果你想要的话这并不难添加（但它不在你应该为作业实现的接口中）。

因为沿单个轴求和即使对于紧凑数组也可能有点棘手，Python 中的这些函数通过将最后一个轴排列为被归约的轴来简化事情（这就是 NDArray 中 `reduce_view_out()` 函数所做的），然后压缩数组。所以对于你在 C++ 中实现的 `ReduceMax()` 和 `ReduceSum()` 函数，你可以假设输入和输出数组在内存中是连续的，你只需要归约传给 C++ 函数的 `reduce_size` 大小的连续元素。

In [13]:
!export PATH="/home/x1879/miniconda3/envs/dlsys/nvvm/bin:$PATH" && make
reload_nd()
run_test('reduce and cpu')

-- Found pybind11: /home/x1879/miniconda3/envs/dlsys/lib/python3.13/site-packages/pybind11/include (found version "3.0.4")
-- Found cuda, building cuda backend
-- Detected CUDA architecture: 8.9
-- Configuring done (0.1s)
-- Generating done (0.0s)
-- Build files have been written to: /home/x1879/dlsys/hws/hw3/build
make[1]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[2]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
[ 50%] Built target ndarray_backend_cpu
make[3]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
[100%] Built target ndarray_backend_cuda
make[2]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
make[1]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
Using needle backend
模块已重载，可以调试
============================= test session starts ============================

In [14]:
# 提交到 mugrade（需要 API key）
# 取消注释以下行并替换 MY_API_KEY
# !python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cpu_reductions"

## 第 5 部分：CPU 后端 - 矩阵乘法

在 `ndarray_backend_cpu.cc` 中实现以下函数：

* `Matmul()`
* `MatmulTiled()`
* `AlignedDot()`

第一个实现 `Matmul()` 可以使用朴素的三重嵌套 for 循环矩阵乘法算法。然而，`MatmulTiled()` 在以分块形式排列的内存上执行相同的矩阵乘法，即作为连续的 4D 数组
```c++
float[M/TILE][N/TILE][TILE][TILE];
```

在矩阵乘法期间填充正确值之前，确保将输出矩阵初始化为零。

注意 Python `__matmul__` 代码在矩阵乘法的所有大小都可被 `TILE` 整除时已经转换为分块形式，所以你的代码只需要在这种形式下实现乘法。为了使方法高效，你将需要利用（在你实现之后）`AlignedDot()` 函数，这将使编译器能够高效地利用向量操作和适当的缓存。输出矩阵也将是上面的分块形式，Python 后端将处理到普通 2D 数组的转换。

注意为了从你的分块版本获得最大的加速，你可能想在 colab 中使用 clang 编译器而不是 gcc。为此，在构建代码之前运行以下命令。

In [15]:
import pytest
!export PATH="/home/x1879/miniconda3/envs/dlsys/nvvm/bin:$PATH" && make
reload_nd()
run_test('matmul and cpu')

-- Found pybind11: /home/x1879/miniconda3/envs/dlsys/lib/python3.13/site-packages/pybind11/include (found version "3.0.4")
-- Found cuda, building cuda backend
-- Detected CUDA architecture: 8.9
-- Configuring done (0.1s)
-- Generating done (0.0s)
-- Build files have been written to: /home/x1879/dlsys/hws/hw3/build
make[1]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[2]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
[ 50%] Built target ndarray_backend_cpu
make[3]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
[100%] Built target ndarray_backend_cuda
make[2]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
make[1]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
Using needle backend
模块已重载，可以调试
============================= test session starts ============================

In [16]:
# 提交到 mugrade（需要 API key）
# 取消注释以下行并替换 MY_API_KEY
# !python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cpu_matmul"

In [17]:
# 使用 clang++ 编译以获得更好的性能
!export CXX=/usr/bin/clang++ && make

-- Found pybind11: /home/x1879/miniconda3/envs/dlsys/lib/python3.13/site-packages/pybind11/include (found version "3.0.4")
-- Found cuda, building cuda backend
-- Detected CUDA architecture: 8.9
-- Configuring done (0.1s)
-- Generating done (0.0s)
-- Build files have been written to: /home/x1879/dlsys/hws/hw3/build
make[1]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[2]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
[ 50%] Built target ndarray_backend_cpu
make[3]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
[100%] Built target ndarray_backend_cuda
make[2]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
make[1]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'


## 第 6 部分：CUDA 后端 - Compact 和 setitem

在 `ndarray_backend_cuda.cu` 中实现以下函数：
* `Compact()`
* `EwiseSetitem()`
* `ScalarSetitem()`


对于这部分，你将在 CUDA 后端中实现 compact 和 setitem 调用。这与 C++ 版本非常相似，然而，取决于你如何实现该函数，也可能存在一些实质性差异。我们特别想强调 C++ 和 CUDA 实现之间的一些区别。

首先，与 CUDA 后端代码中实现的示例函数一样，对于上面所有的函数，你实际上需要实现两个函数：你将从 Python 调用的基本函数，以及实际执行计算的相应 CUDA 内核。在大多数情况下，我们在 `ndarray_backend_cuda.cu` 文件中只提供'基本'函数的原型，你将需要自己定义和实现内核函数。然而，要了解这些如何工作，对于 `Compact()` 调用，我们为你提供了_完整的_ `Compact()` 调用和 `CompactKernel()` 调用的函数原型。

你可能会注意到 `CudaVec` 结构的看似奇怪的使用，这是一个用于传递 shape/stride 参数的结构。在 C++ 版本中，我们使用 STL `std::vector` 变量来存储这些输入（在基本 `Compact()` 调用中也是如此），但 CUDA 内核不能操作 STL 向量，所以需要其他东西。此外，虽然我们_可以_将向量转换为普通 CUDA 数组，但这会相当麻烦，因为我们需要调用 `cudaMalloc()`，将参数作为整数指针传递，然后在调用后释放它们。当然，对于数组的实际底层数据需要这样的内存管理，但仅仅为了传递可变大小的小型 shape/stride 值元组似乎有些过度。解决方案是创建一个具有数组可以拥有的最大维度数'最大化'大小的结构，然后只是将实际的 shape/stride 数据存储在这些字段的第一个条目中。所有这些都是由包含的 `CudaVec` 结构和 `VecToCuda()` 函数完成的，你可以直接使用这些为所有需要将 shape/strides 传递给内核本身的 CUDA 内核。

C++ 和 CUDA 实现 `Compact()` 之间的另一个（更概念性的）大区别是，在 C++ 中你通常会顺序遍历非紧凑数组的元素，这允许你在计算紧凑和非紧凑数组之间的相应索引时执行一些优化。在 CUDA 中，你不能这样做，需要实现能够直接从紧凑数组中的索引映射到步长数组中的索引的代码。

如前所述，我们建议你以这样的方式实现代码：它可以在 `Compact()` 和 `Setitem()` 调用之间轻松重用。简短说明，记住如果你想从内核代码调用（单独的、非内核的）函数，你需要将其定义为 `__device__` 函数。在 CUDA 中，`__global__` 用于定义在 GPU 上运行并从 CPU 主机调用的函数，而 `__device__` 定义仅在 GPU 上运行并从其他 GPU 函数代码调用的函数。

In [18]:
import pytest
!export PATH="/home/x1879/miniconda3/envs/dlsys/nvvm/bin:$PATH" && make
reload_nd()
run_test('(setitem or compact) and cuda')

-- Found pybind11: /home/x1879/miniconda3/envs/dlsys/lib/python3.13/site-packages/pybind11/include (found version "3.0.4")
-- Found cuda, building cuda backend
-- Detected CUDA architecture: 8.9
-- Configuring done (0.1s)
-- Generating done (0.0s)
-- Build files have been written to: /home/x1879/dlsys/hws/hw3/build
make[1]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[2]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
[ 50%] Built target ndarray_backend_cpu
make[3]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
[100%] Built target ndarray_backend_cuda
make[2]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
make[1]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
Using needle backend
模块已重载，可以调试
============================= test session starts ============================

In [19]:
# 提交到 mugrade（需要 API key）
# 取消注释以下行并替换 MY_API_KEY
# !python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cuda_compact_setitem"

## 第 7 部分：CUDA 后端 - 逐元素和标量操作

在 `ndarray_backend_cuda.cu` 中实现以下函数：

* `EwiseMul()`, `ScalarMul()`
* `EwiseDiv()`, `ScalarDiv()`
* `ScalarPower()`
* `EwiseMaximum()`, `ScalarMaximum()`
* `EwiseEq()`, `ScalarEq()`
* `EwiseGe()`, `ScalarGe()`
* `EwiseLog()`
* `EwiseExp()`
* `EwiseTanh()`

同样，我们不提供这些函数原型，欢迎你使用 C++ 模板或宏来使实现更紧凑。你还需要在实现每个函数后取消注释 Pybind11 代码的适当区域。

In [20]:
import pytest
!export PATH="/home/x1879/miniconda3/envs/dlsys/nvvm/bin:$PATH" && make
reload_nd()
run_test( '(ewise_fn or ewise_max or log or exp or tanh or (scalar and not setitem)) and cuda')

-- Found pybind11: /home/x1879/miniconda3/envs/dlsys/lib/python3.13/site-packages/pybind11/include (found version "3.0.4")
-- Found cuda, building cuda backend
-- Detected CUDA architecture: 8.9
-- Configuring done (0.1s)
-- Generating done (0.0s)
-- Build files have been written to: /home/x1879/dlsys/hws/hw3/build
make[1]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[2]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
[ 50%] Built target ndarray_backend_cpu
make[3]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
[100%] Built target ndarray_backend_cuda
make[2]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
make[1]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
Using needle backend
模块已重载，可以调试
============================= test session starts ============================

In [21]:
# 提交到 mugrade（需要 API key）
# 取消注释以下行并替换 MY_API_KEY
# !python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cuda_ops"

## 第 8 部分：CUDA 后端 - 归约操作


在 `ndarray_backend_cuda.cu` 中实现以下函数：

* `ReduceMax()`
* `ReduceSum()`

你可以采取相当简单的方法，只是为每个单独的归约项使用一个单独的 CUDA 线程：即如果有一个 100 x 20 的数组，你沿第二个维度归约，你可以有 100 个线程，每个线程单独处理自己的 20 维数组。这对于 `.max(axis=None)` 调用特别低效，但我们暂时不担心这个。如果你想要更工业级的实现，你可以使用分层机制，首先在某个较小的范围内聚合，然后有一个辅助函数在_这些_归约数组上聚合，等等。但这不是通过测试所必需的。

In [22]:
import pytest
!export PATH="/home/x1879/miniconda3/envs/dlsys/nvvm/bin:$PATH" && make
reload_nd()
run_test('reduce and cuda')

-- Found pybind11: /home/x1879/miniconda3/envs/dlsys/lib/python3.13/site-packages/pybind11/include (found version "3.0.4")
-- Found cuda, building cuda backend
-- Detected CUDA architecture: 8.9
-- Configuring done (0.1s)
-- Generating done (0.0s)
-- Build files have been written to: /home/x1879/dlsys/hws/hw3/build
make[1]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[2]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
[ 50%] Built target ndarray_backend_cpu
make[3]: Entering directory '/home/x1879/dlsys/hws/hw3/build'
make[3]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
[100%] Built target ndarray_backend_cuda
make[2]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
make[1]: Leaving directory '/home/x1879/dlsys/hws/hw3/build'
Using needle backend
模块已重载，可以调试
============================= test session starts ============================

In [23]:
# 提交到 mugrade（需要 API key）
# 取消注释以下行并替换 MY_API_KEY
# !python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cuda_reductions"

## 第 9 部分：CUDA 后端 - 矩阵乘法

在 `ndarray_backend_cuda.cu` 中实现以下函数：

* `Matmul()`

最后，作为你的最后一个练习，你将在 GPU 上实现矩阵乘法。你这里的实现可以大致遵循课堂上的演示。虽然你可以使用相当朴素的代码通过测试（即你可以只为矩阵中的每个 (i,j) 位置使用一个单独的线程），但高效地进行矩阵乘法（使其实际上比 CPU 版本更快）需要协作获取和课堂上介绍的块共享内存寄存器分块。尝试使用这些方法实现，看看你能让你的代码比 C++（或 numpy）后端快多少。

In [25]:
import pytest

# !export PATH="/home/x1879/miniconda3/envs/dlsys/nvvm/bin:$PATH" && make
# reload_nd()
run_test('matmul and cuda')

============================= test session starts ==============================
collecting ... Using needle backend
collected 136 items / 126 deselected / 10 selected

tests/hw3/test_ndarray.py::test_matmul[16-16-16-cuda] PASSED
tests/hw3/test_ndarray.py::test_matmul[8-8-8-cuda] PASSED
tests/hw3/test_ndarray.py::test_matmul[1-2-3-cuda] PASSED
tests/hw3/test_ndarray.py::test_matmul[3-4-5-cuda] PASSED
tests/hw3/test_ndarray.py::test_matmul[5-4-3-cuda] PASSED
tests/hw3/test_ndarray.py::test_matmul[64-64-64-cuda] PASSED
tests/hw3/test_ndarray.py::test_matmul[72-72-72-cuda] PASSED
tests/hw3/test_ndarray.py::test_matmul[72-73-74-cuda] PASSED
tests/hw3/test_ndarray.py::test_matmul[74-73-72-cuda] PASSED
tests/hw3/test_ndarray.py::test_matmul[128-128-128-cuda] PASSED

====================== 10 passed, 126 deselected in 0.45s ======================



In [ ]:
# 提交到 mugrade（需要 API key）
# 取消注释以下行并替换 MY_API_KEY
# !python3 -m mugrade submit "$MY_API_KEY" "$HW3_NAME" -k "ndarray_cuda_matmul"